# Formatting Envision files

We will be formatting the envision outputs

In [63]:
import polars as pl
from pathlib import Path
import io

raw_envision_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA")

data_headers = ('Plate', 'Barcode', 'Well')
end_col = 6

In [64]:
def parse_envision(path: Path):
    with open(path, 'r') as f:
        lines = [l.strip() for l in f.readlines()]
    
    header_idx = None
    
    for i, line in enumerate(lines):
        line = line.strip()
        cols = [c.strip() for c in line.split(",")]
        if data_headers[0] in cols and data_headers[1] in cols and data_headers[2] in cols:
            header_idx = i
            break
    
    data_lines = [lines[header_idx]]
    
    for line in lines[header_idx + 1:]:
        if not line.strip():
            break
        first_col = line.split(',')[0].strip()
        if not first_col.isdigit():
            break
        data_lines.append(line)
        
    cleaned = "\n".join(data_lines).replace('=""', '')
    
    return pl.read_csv(io.StringIO(cleaned), null_values=['', 'NA'])
        

In [65]:
dfs = []

for i, file in enumerate(sorted(raw_envision_path.iterdir(), reverse=False), start=1):
    df = (parse_envision(file)
        .drop(["MeasTime", "CalcResultI_duplicated_0",	"Signal_duplicated_0", "MeasTime_duplicated_0", "_duplicated_0", "_duplicated_1", "", "Barcode"])
        .with_columns(pl.lit(i).alias("Timepoint"))
    )
    
    dfs.append(df)

final_formatted_df = pl.concat(dfs)

display(final_formatted_df)
final_formatted_df.write_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_formatted/2026-04-01._.FORMATTED.csv")

Plate,Well,CalcResultI,Signal,Flashes/Time,Timepoint
i64,str,i64,i64,f64,i32
1,"""A01""",5000,1000,0.2,1
1,"""A02""",5100,1020,0.2,1
1,"""A03""",5400,1080,0.2,1
1,"""A04""",4800,960,0.2,1
1,"""A05""",4620,924,0.2,1
…,…,…,…,…,…
1,"""P20""",5000,1000,0.2,235
1,"""P21""",5300,1060,0.2,235
1,"""P22""",5080,1016,0.2,235
